In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color="red" size="10">ch14. 웹데이터 수집II</b></font>

# 1절. selenium을 이용한 동적 웹크롤링 문법
- https://selenium-python.readthedocs.io
- `pip install selenium`(아나콘다 프롬프트에서 버전 확인)
    - 경고 무시 => pip install --upgrade requests (requests를 최신버전으로 upgrade)하거나
                 conda pip install urllib3==1.26.18 
- selenium버전 : 4.47 / requests버전 : 2.28.1 / urllib3버전 : 2.7.0

In [12]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

In [13]:
dv = webdriver.Chrome()
dv.get('http://python.org')

In [14]:
elem = dv.find_element(By.NAME,'q')
#By.CLASS_NAME, By.ID, By.CSS_SELECTOR, BY.TAG_NAME
# a태그에서 By.LINK_TEXT, by.PARTIAL_LINK_TEXT
elem.clear()
elem.send_keys('pycon')
elem.send_keys(Keys.RETURN)#enter

In [10]:
dv.get('http://python.org')
elem = dv.find_element(By.NAME,'q')
elem.send_keys(Keys.CONTROL,'a')#ctrl+a
elem.send_keys('pycon')
btn_elem = dv.find_element(By.CSS_SELECTOR,'button#submit')
btn_elem.click()

In [18]:
result_list = dv.find_elements(By.CSS_SELECTOR,'li>h3>a')
# len(result_list)
for result in result_list:
    print("{}-{}".format(result.text,result.get_attribute('href')))

PSF PyCon Trademark Usage Policy-https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette)-https://www.python.org/events/python-events/378/
PyCon Australia 2013-https://www.python.org/events/python-events/57/
PyCon AU 2019-https://www.python.org/events/python-events/776/
PyCon NL 2025-https://www.python.org/events/python-events/2084/
PyCon Australia 2014-https://www.python.org/events/python-events/10/
PyCon Ireland 2012-https://www.python.org/events/python-events/76/
PyCon Ireland 2016-https://www.python.org/events/python-events/429/
PyCon Ireland 2022-https://www.python.org/events/python-events/1320/
PyCon Australia 2014-https://www.python.org/events/python-events/1447/
PyCon Ireland 2023-https://www.python.org/events/python-events/1568/
PyCon Ireland 2024-https://www.python.org/events/python-events/1862/
PyCon APAC 2025-https://www.python.org/events/python-events/1879/
PyCon AU 2018-https://www.python.org/events/python-events/696/
PyCon APAC 2022-https://www.python.

In [22]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source,'html.parser')
result_list = soup.select('li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{}-{}".format(result.text,result.attrs.get('href')))

PSF PyCon Trademark Usage Policy-/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette)-/events/python-events/378/
PyCon Australia 2013-/events/python-events/57/


In [25]:
from urllib.parse import urlparse
#https://www.python.org/search/?q=pycon&submit=
current_url = dv.current_url
print('현재 url:', current_url)
result_parse = urlparse(current_url)
print('url parsing 결과:',result_parse)
domain = f'{result_parse.scheme}://{result_parse.netloc}'
domain = "{}://{}".format(result_parse.scheme,result_parse.netloc)
print('현재 domain:',domain)

현재 url: https://www.python.org/search/?q=pycon&submit=
url parsing 결과: ParseResult(scheme='https', netloc='www.python.org', path='/search/', params='', query='q=pycon&submit=', fragment='')
현재 domain: https://www.python.org


In [26]:
soup = BeautifulSoup(dv.page_source,'html.parser')
result_list = soup.select('li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{}-{}".format(result.text,domain+result.attrs.get('href')))

PSF PyCon Trademark Usage Policy-https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette)-https://www.python.org/events/python-events/378/
PyCon Australia 2013-https://www.python.org/events/python-events/57/


In [27]:
dv.close()#브라우저 종료

# 2절 동적 웹크롤링 예제
## 2-1다음 뉴스 검색 예제

In [42]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time
news_list=[]# 뉴스제목과 뉴스link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5)#다음패이지가 다 쯜때까지 0.5초 대기

query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME,'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR,'button[type=submit]').click()
time.sleep(2)#패이지 로딩될시간동안 대기하기
#뉴스 탭 클릭
# driver.find_element(By.CSS_SELECTOR,'ul.list_tab>')[1].click()
driver.find_element(By.LINK_TEXT,'뉴스').click()

검색할 단어는?야구


In [46]:
bodies = driver.find_elements(By.CSS_SELECTOR,'strong.tit-g.clamp-g')
# len(bodies)
for body in bodies:
    a=body.find_element(By.TAG_NAME,'a')
    title = a.text
    link = a.get_attribute('href')
#     print(title,link)
    news_list.append([title,link])

In [51]:
page_nav = driver.find_element(By.CLASS_NAME,'inner_paging')
# page_nav.text
nex_page = page_nav.find_element(By.LINK_TEXT,'3')# a태그의 text가 2인 a테그
nex_page.click()

In [7]:
import pandas as pd
pd.DataFrame(news_list,columns=['뉴스제목','링크']).shape

(30, 2)

## 2-2 다음 뉴스 패이징 처리
- 위의 예제를 이용하여 원하는 패이지 만큼 뉴스 검색 결과를 받아오기

In [9]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time
news_list=[]# 뉴스제목과 뉴스link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5)#다음패이지가 다 쯜때까지 0.5초 대기

query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME,'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR,'button[type=submit]').click()
time.sleep(2)#패이지 로딩될시간동안 대기하기
#뉴스 탭 클릭
# driver.find_element(By.CSS_SELECTOR,'ul.list_tab>')[1].click()
driver.find_element(By.LINK_TEXT,'뉴스').click()

pages = int(input('몇패이지 크롤링 할까요?'))
for page in range(1,pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR,'strong.tit-g.clamp-g')
    for body in bodies:   
        a=body.find_element(By.TAG_NAME,'a')
        title = a.text
        link = a.get_attribute('href')
#     print(title,link)
        news_list.append([title,link])
    page_nav = driver.find_element(By.CLASS_NAME,'inner_paging')
# page_nav.text
    nex_page = page_nav.find_element(By.LINK_TEXT,str(page+1))# a태그의 text가 2인 a테그
    nex_page.click()
    time.sleep(2)
# driver.close()
news_df = pd.DataFrame(news_list,columns=['title','link'])
display(news_df.head())
print(news_df.shape)

검색할 단어는?농구
몇패이지 크롤링 할까요?4


,title,link
0,[KBL유소년] '유소년 꿈의 무대' DB손해보험 2026 KBL 유스클럽 농구대회...,http://v.daum.net/v/20260818170439835
1,"라이즈 신곡 ‘라이즈 오버’, 일본 고교 농구 ‘윈터컵’ 공식 응원가 선정",http://v.daum.net/v/20260818111128478
2,"[스포츠+] NBA 꿈꾸는 한국 농구 에이스, 이현중② ""두려움 없이 도전하겠다""",http://v.daum.net/v/20260818103114116
3,"1964년부터 ‘개근’, 대한민국 女농구의 놀라운 62년…‘세계 최강’ 미국과 공동...",http://v.daum.net/v/20260818095550218
4,"[오피셜] 여자농구 하나은행, 트레이드 통해 유망주 가드 김수인 영입",http://v.daum.net/v/20260818141740994


(40, 2)


## 2-3 맞춤법 검사기

In [26]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

In [27]:
driver = webdriver.Chrome()

In [28]:
driver.get('https://www.naver.com/')
time.sleep(1)
elem = driver.find_element(By.ID,'query')
elem.send_keys(Keys.CONTROL,'a')
elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(1)
textarea = driver.find_element(By.CLASS_NAME,'txt_gray')
textarea.clear() # input이나  textarea
textarea.send_keys('안뇽하새요. 방갑습니다.맛잇는 점심시간 되세요')
btn = driver.find_element(By.CLASS_NAME,'btn_check')
btn.click()
time.sleep(2)
result = driver.find_element(By.CSS_SELECTOR,'p._result_text.stand_txt').text
print(result)
driver.close()

안녕하세요. 반갑습니다. 맛있는 점심시간 보내세요


### 맞춤법 검사전. txt파일을 맞춤법 검사후.txt로 파일 출력

In [7]:
# fp = open('data/ch14_맞춤법 검사_전.txt','r',encoding='utf-8')
# text = fp.read()
# fp.close()
with open('data/ch14_맞춤법 검사_전.txt','r',encoding='utf-8')as fp:
    text = fp.read()
ready_text_list =[]#300자 기준으로 문장단위로 나눠진 text list
while len(text)>=300:
  
    temp = text[:300]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)  
print([len(read_text)for read_text in ready_text_list])

[263, 251, 264, 291, 160]


['야구는 세계적으로 많은 사랑을 받는 스포츠로, 오랜 역사와 전통을 가지고 있다. 야구의 정확한 기원에 대해서는 여러 가지 의견이 있지만, 일반적으로 영국의 전통 스포츠인 크리켓과 라운더스에서 영향을 받은 것으로 알려져 있다. 이러한 경기 방식이 북아메리카로 전해지면서 새로운 형태의 스포츠로 발전했고, 오늘날 우리가 알고 있는 야구의 모습이 만들어졌다.\n\n19세기 초 미국에서는 다양한 형태의 야구 경기가 이루어졌지만, 지역마다 규칙이 달라 통일된 경기 운영이 어려웠다.',
 ' 이러한 문제를 해결하기 위해 1845년 미국의 알렉산더 카트라이트가 뉴욕 니커보커 야구단의 규칙을 정리하면서 현대 야구의 기초가 마련되었다. 그는 경기장의 형태와 선수 수, 아웃 규칙 등을 체계적으로 정비했으며, 이는 현재 야구 규칙의 기본이 되었다.\n\n1846년에는 뉴저지에서 최초의 공식 야구 경기가 열렸으며, 이후 야구는 미국 전역으로 빠르게 확산되었다. 특히 남북전쟁이 끝난 후 군인들이 전국 각지로 이동하면서 야구가 더욱 널리 보급되었다.',
 ' 1869년에는 세계 최초의 프로 야구단인 신시내티 레드스타킹스가 창단되었고, 이를 계기로 프로야구 시대가 시작되었다. 이후 1903년 메이저리그 월드시리즈가 개최되면서 미국 야구는 더욱 큰 인기를 얻었다.\n\n20세기에 들어서면서 야구는 미국을 넘어 전 세계로 확산되었다. 일본은 1870년대에 미국인 교사를 통해 야구를 받아들였으며, 이후 학생과 직장인을 중심으로 야구 문화가 발전했다. 중남미 국가들도 야구를 적극적으로 수용하여 현재는 세계적인 야구 강국으로 성장했다.',
 ' 특히 쿠바와 도미니카공화국, 베네수엘라는 수많은 유명 선수를 배출하며 국제 야구 발전에 큰 영향을 미쳤다.\n\n우리나라에 야구가 처음 소개된 시기는 1905년경으로 알려져 있다. 미국인 선교사와 교육자들에 의해 야구가 전파되었으며, 학교를 중심으로 점차 보급되었다. 이후 고교야구가 큰 인기를 얻으면서 많은 국민의 관심을 받게 되었고, 1982년 한국 

In [12]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
import time
driver = webdriver.Chrome()
driver.get('https://www.naver.com/')
time.sleep(1)
elem = driver.find_element(By.ID,'query')

elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(1)
textarea = driver.find_element(By.CLASS_NAME,'txt_gray')
results = '' # 맞춤법 검사후
for idx, ready_text in enumerate(ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear() # input이나  textarea
    textarea.send_keys(ready_text)
    btn = driver.find_element(By.CLASS_NAME,'btn_check')
    btn.click()
    time.sleep(1)
    # result = driver.find_element(By.CSS_SELECTOR,'p._result_text.stand_txt').text
    soup = BeautifulSoup(driver.page_source,'html.parser')
    result = soup.select_one('p._result_text.stand_txt').text
    results += result

driver.close()

검사중...1/5
검사중...2/5
검사중...3/5
검사중...4/5
검사중...5/5


In [17]:
# 맞춤법 검사 결과(results)를 파일 출력
with open('data/ch14_맞춤법검사후.txt','w') as fp:
    fp.write(results)

# 3절. 연습문제
- https://papago.naver.com/ 를 통해서 "data/ch14_맞춤법후.txt" 파일의 영문으로 번역하여 파일 출력

In [47]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
import time

In [48]:
driver = webdriver.Chrome()
driver.get('https://papago.naver.com/')
time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME,'entry-popup-module-scss-module__UZIxta__close')
if btn:
    btn.click()
    print('축하해달라는 창 닫음')
else:
    print('축하창 안뜸')
input_elem = driver.find_element(By.CLASS_NAME,
                                 'text-translator-module-scss-module__CYJRkW__text-editor')
input_elem.send_keys("안녕하세요.반갑습니다.내일부터는 데이터 배이스 정식 수업입니다.")
time.sleep(0.)
result = driver.find_elements(By.CLASS_NAME,
                             'text-editor-module-scss-module__gKzuvW__dynamic-md')[1].text
print(result)
driver.close()

축하해달라는 창 닫음



In [65]:
with open('data/ch14_맞춤법 검사전quiz.txt','r',encoding='utf-8')as fp:
    text = fp.read()
ready_text_list =[]
while len(text)>=3000:
  
    temp = text[:3000]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)  
print([len(read_text)for read_text in ready_text_list])

[2649]


In [68]:
driver = webdriver.Chrome()
driver.get('https://papago.naver.com/')
time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME,'entry-popup-module-scss-module__UZIxta__close')
if btn:
    btn.click()
#     print('축하해달라는 창 닫음')
# else:
#     print('축하창 안뜸')
input_elem = driver.find_element(By.CLASS_NAME,
                                 'text-translator-module-scss-module__CYJRkW__text-editor')
results = ''

for idx, ready_text in enumerate(ready_text_list):
    print(f'번역중...{idx+1}/{len(ready_text_list)}')
    input_elem.send_keys(ready_text)
    time.sleep(2)
    result = driver.find_elements(By.CSS_SELECTOR,
                                 'div[class^=text-editor-module-scss-module]')[1].text
    results +=result
# print(result)
# driver.close()
with open('data/ch14_자동화영어번역본.txt','w',encoding='utf-8') as f:
    f.write(results)

검사중...1/1


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

# query = 'AI'
query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기
# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

pages = int(input('몇 페이지 크롤링 할까요?'))
for page in range(1, pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
    for body in bodies:
        a = body.find_element(By.TAG_NAME, 'a')
        title = a.text
        link  = a.get_attribute('href')
        # print(title, link)
        news_list.append([title, link])
    page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
    nex_page = page_nav.find_element(By.LINK_TEXT, str(page+1)) # a태그의 text가 2인 a태그
    nex_page.click()
    time.sleep(2)
driver.close()
news_df = pd.DataFrame(news_list, columns=['title','link'])
display(news_df.head())
print(news_df.shape)


검색할 단어는?야구
몇 페이지 크롤링 할까요?2


NameError: name 'pd' is not defined